# Week 9 — From clusters to classes: supervised learning on the Florida images
### Turning last week's SOM clusters into labels, and training models that learn them

Last week the SOM grouped the images **without being told what to look for** — *unsupervised* learning.
Today you decide what the groups **mean**: you look at them, merge or exclude some, and give them names. These names become **labels**, and we train models that learn to predict the label of an image from its numbers — *supervised* learning.

```
 1  LOOK     reload the Week 8 CSVs  ·  last week's clusters  ·  finer sub-clusters of the same SOM
        ▼
 2  LABEL    interactive panel: name, merge or exclude groups   →  new column `label` in the CSV
        ▼
 3  LEARN    baseline · decision tree · logistic regression · random forest · k-NN · gradient boosting
             · neural network (MLP)  →  compare the models  ·  look at their mistakes
             optional: predict one view from the other (satellite ↔ street view) · a first tiny CNN
```

| Section | What you do | Saved in your Drive (`Week9/`) | Time |
|---|---|---|---|
| **0** Setup | mount Drive, find last week's results, settings | – | 5 min |
| **1** Look | last week's clusters and their sub-clusters | – | 8 min |
| **2** Label | interactive panel → new `label` column | `labels/` | 10 min |
| **3** Learn | split the data · readable models · neural network · comparison · mistakes | `figures/` | 35 min (+ optional parts) |
| **4** Wrap-up | predictions into the CSV, saved model, exercises | `labels/`, `models/` | 2 min |

**Before you start**
* **No GPU, no API key, no installation**: everything runs on the normal Colab CPU runtime in a few minutes, and this notebook makes **no paid requests**.
* You need **last week's results** in your Drive (`Colab Notebooks/ML_for_Architects/Week8/`): the CSVs with the SOM labels, the saved SOM and the images.
  If your Week 8 run is incomplete, add the instructor's shared folder **`Week8_shared_data`** to your Drive (open the shared link → *Organize → Add shortcut* → *My Drive*); the notebook then uses it automatically.
* Last week's files are only **read**: everything this notebook writes goes into a new folder `Week9/`.

---
# 0 · Setup

In [ ]:
# 0.1  Imports, Google Drive and folders (everything used in this notebook is pre-installed on Colab)
import io, re, json, math, time, copy, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from IPython.display import Image as IPyImage
import ipywidgets as W
from tqdm.auto import tqdm

%config InlineBackend.figure_formats = ["jpeg"]
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
    COURSE_DIR = MY_DRIVE / "Colab Notebooks" / "ML_for_Architects"
else:  # running on your own computer: the same folders, next to this notebook
    MY_DRIVE = Path.cwd()
    COURSE_DIR = Path.cwd() / "ML_for_Architects"

OWN_WEEK8_DIR = COURSE_DIR / "Week8"                 # your results from last week
SHARED_WEEK8_DIR = MY_DRIVE / "Week8_shared_data"    # the instructor's results (a shortcut in "My Drive")
WEEK9_DIR = COURSE_DIR / "Week9"                     # everything this notebook writes
LABEL_DIR, MODEL_DIR, FIG_DIR, CACHE_DIR = (WEEK9_DIR / d for d in ("labels", "models", "figures", "cache"))
for d in (LABEL_DIR, MODEL_DIR, FIG_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print("Running on Colab" if IN_COLAB else "Running locally", "| this notebook writes to:", WEEK9_DIR)

### 0.2 Settings
The main cell to edit (on Colab the settings appear as menus next to the code).

* `SOURCE` — which images to work with. **Street view** is the main example. **ArchDaily** lets you build an *exterior / interior / drawing* classifier, and **satellite** holds a surprise (exercise 2 at the end).
* `WEEK8_DATA` — `"auto"` uses your own Week 8 results if they are complete, otherwise the instructor's shared ones. Force one with `"own"` or `"shared"`.
* `MERGE_LEVEL` — what you will turn into classes in Section 2:
  * `"clusters"`: last week's SOM clusters **as they are** (keep them, or merge some);
  * `"sub-clusters"`: the same SOM cut into `N_SUBCLUSTERS` smaller groups, each lying inside one of last week's clusters — a second chance to decide what belongs together. If you change nothing in Section 2, you get last week's clusters back.

In [ ]:
#@title 0.2 Settings
SOURCE = "streetview"          #@param ["streetview", "satellite", "archdaily"]
WEEK8_DATA = "auto"            #@param ["auto", "own", "shared"]
MERGE_LEVEL = "sub-clusters"   #@param ["sub-clusters", "clusters"]
N_SUBCLUSTERS = 8              #@param {type:"slider", min:3, max:16, step:1}

In [ ]:
# 0.3  Find last week's results: the CSVs with the SOM labels, the saved SOM and the images
SOURCES = ["streetview", "satellite", "archdaily"]
GOOGLE_LOGO_PX = 24      # Google logo strip at the bottom of satellite / street view images (cut off, as in Week 8)
# readable indicators that make sense for each source (the same choice as in Week 8)
INT_COLS_BY_SOURCE = {
    "streetview": ["int_vegetation", "int_sky", "int_building", "int_road", "int_vehicle", "int_edge_density",
                   "int_vertical_edges", "int_colourfulness"],
    "satellite": ["int_vegetation", "int_green_pixels", "int_grey_pixels", "int_blue_pixels", "int_edge_density",
                  "int_brightness", "int_colourfulness"],
    "archdaily": ["int_interior", "int_wall", "int_sky", "int_vegetation", "int_building", "int_edge_density",
                  "int_vertical_edges", "int_colourfulness"],
}


def week8_status(folder, source):
    """What does a Week 8 folder contain for one source?"""
    emb_dir = folder / "02_features" / "embeddings" / source
    csvs = sorted(emb_dir.glob("*.csv")) if emb_dir.is_dir() else []
    labelled = bool(csvs) and all("som_cluster" in pd.read_csv(f, nrows=0).columns for f in csvs)
    return {"CSVs": len(csvs), "SOM labels": labelled,
            "saved SOM": (folder / "03_som" / "models" / f"{source}_som_weights.npy").exists(),
            "images": (folder / "01_images" / source).is_dir()}


def choose_week8_dir(source, mode):
    """Your own Week 8 folder if it has the SOM labels of `source` (mode "auto"), otherwise the shared one."""
    own, shared = week8_status(OWN_WEEK8_DIR, source), week8_status(SHARED_WEEK8_DIR, source)
    if mode == "own" or (mode == "auto" and own["SOM labels"]):
        folder, status = OWN_WEEK8_DIR, own
    elif mode == "shared" or (mode == "auto" and shared["SOM labels"]):
        folder, status = SHARED_WEEK8_DIR, shared
    else:
        folder, status = OWN_WEEK8_DIR, own
    if not status["SOM labels"]:
        raise FileNotFoundError(
            f"No {source} CSVs with SOM labels in {folder / '02_features' / 'embeddings' / source}\n"
            "-> finish Week 8 (Sections 2 and 3), or add the instructor's folder 'Week8_shared_data' to My Drive "
            "(open the shared link, Organize > Add shortcut > My Drive) and use WEEK8_DATA = 'auto'.")
    return folder


display(pd.DataFrame({(where, src): week8_status(folder, src)
                      for where, folder in [("own", OWN_WEEK8_DIR), ("shared", SHARED_WEEK8_DIR)]
                      for src in SOURCES}).T)
WEEK8_DIR = choose_week8_dir(SOURCE, WEEK8_DATA)
print(f"-> using the {SOURCE} results in {WEEK8_DIR}")

In [ ]:
# 0.4  Helpers: pictures, short descriptions of groups, colours
CITY_ORDER = ["Gainesville", "Tallahassee", "Jacksonville", "Orlando", "Tampa", "Miami", "Saint-Malo"]
CLUSTER_COLORS = [mpl.colors.to_hex(c) for c in mpl.colormaps["tab10"].colors]   # the cluster colours of Week 8


def load_rgb(path):
    """Open an image file as RGB (the file is closed afterwards)."""
    with Image.open(path) as im:
        return im.convert("RGB")


def find_image_dir(df, candidates):
    """The first folder that contains (almost) all images of the table."""
    sample = df.relpath.dropna().sample(min(30, len(df)), random_state=SEED)
    for folder in candidates:
        if folder.is_dir() and np.mean([(folder / p).exists() for p in sample]) >= 0.9:
            return folder
    return None


def show_images(paths, titles=None, ncols=6, size=2.3, suptitle=None):
    """Pictures in a grid."""
    paths = list(paths)
    if not paths:
        print("(no images to show)")
        return
    nrows = math.ceil(len(paths) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 1.05), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, p in enumerate(paths):
        ax = axes.ravel()[i]
        try:
            im = load_rgb(p)
            im.thumbnail((300, 300))
            ax.imshow(im)
        except Exception:
            ax.text(0.5, 0.5, "image not found", ha="center", va="center", fontsize=8, color="#999")
        if titles is not None:
            ax.set_title(str(titles[i]), fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()


def gallery_rows(rows, n=8, size=1.75, suptitle=None):
    """rows = [(title, [paths]), …]: one row of pictures per entry, the title above the row."""
    height = len(rows) * (size * 0.78 + 0.32) + (0.8 if suptitle else 0.4)
    fig, axes = plt.subplots(len(rows), n, figsize=(n * size, height), squeeze=False)
    for r, (title, paths) in enumerate(rows):
        for c in range(n):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(paths):
                try:
                    im = load_rgb(paths[c])
                    im.thumbnail((240, 240))
                    ax.imshow(im)
                except Exception:
                    ax.text(0.5, 0.5, "image\nnot found", ha="center", va="center", fontsize=8, color="#999")
        axes[r, 0].set_title(title, loc="left", fontsize=9, fontweight="bold")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1 - 0.08 / height, va="top")
    # fixed margins (an automatic layout would shrink the pictures to make room for the long row titles)
    fig.subplots_adjust(left=0.005, right=0.995, bottom=0.01, top=1 - (0.75 if suptitle else 0.35) / height,
                        wspace=0.04, hspace=0.42)
    plt.show()


def typical_first(idx, X):
    """Indices sorted from the most typical image (closest to the group's average embedding) to the least typical."""
    idx = np.asarray(idx)
    centre = X[idx].mean(axis=0)
    return idx[np.argsort(np.linalg.norm(X[idx] - centre, axis=1))]


def example_paths(idx, n=8):
    """n pictures of a group, spread from the most typical to the least typical."""
    order = typical_first(idx, X_EMB)
    pick = order[np.unique(np.linspace(0, len(order) - 1, min(n, len(order))).round().astype(int))]
    return [IMG_DIR / p for p in DATA.relpath.iloc[pick]]


def describe(mask, n_up=2, n_down=1):
    """What is special about a group of images: readable indicators clearly above (↑) or below (↓) the average."""
    z = ((DATA.loc[mask, READABLE].mean() - DATA[READABLE].mean()) / DATA[READABLE].std()).dropna().sort_values()
    up = [f"{c[4:].replace('_', ' ')} {'↑↑' if v > 1 else '↑'}" for c, v in z[::-1].items() if v > 0.4][:n_up]
    down = [f"{c[4:].replace('_', ' ')} {'↓↓' if v < -1 else '↓'}" for c, v in z.items() if v < -0.4][:n_down]
    return " · ".join(up + down) or "close to the average image"


def shade(color, amount):
    """A lighter version of a colour (amount 0 = same colour, 1 = white)."""
    rgb = np.array(mpl.colors.to_rgb(color))
    return mpl.colors.to_hex(rgb + (1 - rgb) * amount)

---
# 1 · Look again: last week's clusters
Every row of the Week 8 CSVs is one image:

| Columns | Content | Used today for |
|---|---|---|
| `image_id`, `city`, `unit`, `lat`, `lon`, `relpath` | which image, where, which file | splitting the data, showing pictures |
| `emb_000 … emb_099` | the embedding (four lenses fused + PCA) — **the SOM was trained on these** | features (**X**) |
| `int_…` (20 columns) | readable indicators: shares of vegetation, sky, building…, edges, colour | features (**X**), describing groups |
| `som_node`, `som_cluster`, … | where the SOM put the image | the starting point of our labels (**y**) |

In [ ]:
# 1.1  Load the CSVs of all cities of SOURCE into one table
def load_week8_table(folder, source):
    files = sorted((folder / "02_features" / "embeddings" / source).glob("*.csv"))
    df = pd.concat([pd.read_csv(f, dtype={"image_id": str}) for f in files], ignore_index=True)
    return df.drop_duplicates("image_id").reset_index(drop=True).copy()     # .copy(): one compact table (faster)


DATA = load_week8_table(WEEK8_DIR, SOURCE)
if DATA.som_cluster.isna().any():
    print(f"⚠️  {int(DATA.som_cluster.isna().sum())} images have no SOM label and are left out.")
    DATA = DATA.dropna(subset=["som_cluster"]).reset_index(drop=True)
DATA["som_cluster"] = DATA.som_cluster.astype(int)
DATA["som_node"] = DATA.som_node.astype(int)
EMB_COLS = [c for c in DATA.columns if c.startswith("emb_")]
INT_COLS = [c for c in DATA.columns if c.startswith("int_") and DATA[c].std() > 0]
READABLE = [c for c in INT_COLS_BY_SOURCE[SOURCE] if c in INT_COLS] or INT_COLS[:8]
X_EMB = DATA[EMB_COLS].to_numpy(dtype=float)
K_WEEK8 = int(DATA.som_cluster.nunique())
IMG_DIR = find_image_dir(DATA, [WEEK8_DIR / "01_images", OWN_WEEK8_DIR / "01_images", SHARED_WEEK8_DIR / "01_images"])

print(f"{SOURCE}: {len(DATA)} images · {len(EMB_COLS)} emb_ columns · {len(INT_COLS)} int_ columns · "
      f"last week's SOM: {K_WEEK8} clusters")
print("pictures:", IMG_DIR if IMG_DIR else "NOT FOUND -> no galleries, and Section 3.6 is skipped")
table = pd.crosstab(DATA.city, DATA.som_cluster, margins=True, margins_name="all")
table.columns = [c if c == "all" else f"cluster {c}" for c in table.columns]
display(table)

In [ ]:
# 1.2  Reminder: last week's SOM — the most typical image of every node, framed in the colour of its cluster
fig_path = WEEK8_DIR / "03_som" / "figures" / f"{SOURCE}_som_images.png"
if fig_path.exists():
    with Image.open(fig_path) as im:
        im = im.convert("RGB")
        im.thumbnail((900, 900))
        buf = io.BytesIO()
        im.save(buf, "JPEG", quality=85)
    display(IPyImage(data=buf.getvalue(), format="jpeg", width=600))
else:
    print("(figure not found:", fig_path, ")")

In [ ]:
# 1.3  What do last week's clusters look like? Left: the most typical image of the cluster -> right: less typical
if IMG_DIR is None:
    print("No images found -> gallery skipped.")
else:
    rows = []
    for c in sorted(DATA.som_cluster.unique()):
        mask = (DATA.som_cluster == c).to_numpy()
        rows.append((f"cluster {c} · {mask.sum()} images · {describe(mask)}", example_paths(np.where(mask)[0])))
    gallery_rows(rows, suptitle=f"{SOURCE}: last week's {K_WEEK8} SOM clusters")

## 1.4 A second chance: sub-clusters of the same SOM
In Week 8 the SOM nodes were merged into clusters by **hierarchical clustering**: starting with one group per node, the two most similar *neighbouring* groups were merged, again and again, until `k` groups were left. Stopping this merging **earlier** leaves more, smaller groups: the **sub-clusters**. Because they come from the same sequence of merges, every sub-cluster lies completely inside one of last week's clusters — `0a`, `0b`, … are the parts of cluster 0.

Now you decide: are `0a` and `0b` really the same kind of place? Does `1c` belong with `0b` rather than with `1a`? Is a group so mixed — or so odd — that it should be left out?
*(With `MERGE_LEVEL = "clusters"` there are no sub-clusters: Section 2 starts from last week's clusters.)*

In [ ]:
# 1.4  Cut last week's SOM into sub-clusters (Week 8's method: Ward linkage of neighbouring nodes, stopped earlier)
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
from sklearn.metrics import adjusted_rand_score

LETTERS = "abcdefghijklmnopqrstuvwxyz"


def cut_som(weights, k):
    """Merge the nodes of a SOM into k connected groups -> one group number per node (= cluster_nodes() of Week 8)."""
    rows, cols, dim = weights.shape
    model = AgglomerativeClustering(n_clusters=int(k), linkage="ward", connectivity=grid_to_graph(rows, cols))
    return model.fit_predict(weights.reshape(rows * cols, dim))


def make_subclusters(n_sub):
    """Sub-cluster name of every image ('0a', '0b', … = the parts of cluster 0) — or None and the reason."""
    model_dir = WEEK8_DIR / "03_som" / "models"
    w_path = model_dir / f"{SOURCE}_som_weights.npy"
    if not w_path.exists():
        return None, f"the saved SOM ({w_path.name}) was not found"
    weights = np.load(w_path)
    n_nodes = weights.shape[0] * weights.shape[1]
    if DATA.som_node.max() >= n_nodes:
        return None, "the saved SOM does not match the CSVs (different map size)"
    k_week8 = K_WEEK8
    if (model_dir / f"{SOURCE}_som.json").exists():
        k_week8 = int(json.loads((model_dir / f"{SOURCE}_som.json").read_text()).get("n_clusters", K_WEEK8))
    nodes = DATA.som_node.to_numpy()
    agreement = adjusted_rand_score(DATA.som_cluster, cut_som(weights, k_week8)[nodes])
    if agreement < 0.999:   # the saved SOM must give last week's clusters back
        return None, f"the saved SOM does not reproduce the clusters in the CSVs (agreement {agreement:.2f})"
    k_sub = int(min(max(n_sub, k_week8), n_nodes))
    fine = cut_som(weights, k_sub)[nodes]
    names = {}
    for parent in sorted(DATA.som_cluster.unique()):          # name the parts of each cluster a, b, c… (largest first)
        inside = pd.DataFrame({"fine": fine, "node": nodes})[(DATA.som_cluster == parent).to_numpy()]
        stats = inside.groupby("fine").node.agg(["size", "min"]).sort_values(["size", "min"], ascending=[False, True])
        for letter, f in zip(LETTERS, stats.index):
            names[f] = f"{parent}{letter}"
    sub = np.array([names[f] for f in fine])
    if pd.crosstab(sub, DATA.som_cluster.to_numpy()).gt(0).sum(axis=1).max() != 1:
        return None, "the sub-clusters are not nested in last week's clusters"
    return sub, dict(weights=weights, k_sub=k_sub, k_week8=k_week8)


def plot_subcluster_map(weights, k_sub, k_week8):
    """The sub-clusters drawn on the SOM grid, with the borders of last week's clusters."""
    rows, cols, _ = weights.shape
    fine_node = cut_som(weights, k_sub)
    coarse = cut_som(weights, k_week8).reshape(rows, cols)
    name_of = dict(zip(fine_node[DATA.som_node.to_numpy()], DATA.group))       # sub-cluster number -> name
    names = np.array([name_of.get(f, "") for f in fine_node]).reshape(rows, cols)
    hits = np.bincount(DATA.som_node, minlength=rows * cols).reshape(rows, cols)
    fig, ax = plt.subplots(figsize=(0.62 * cols + 1.6, 0.62 * rows + 1.3))
    ax.imshow([[mpl.colors.to_rgb(GROUP_COLORS.get(n, "#eeeeee")) for n in row] for row in names])
    for i in range(rows):
        for j in range(cols):
            ax.text(j, i, f"{names[i, j]}\n{hits[i, j] or ''}", ha="center", va="center", fontsize=7)
            if j + 1 < cols and coarse[i, j] != coarse[i, j + 1]:
                ax.plot([j + 0.5, j + 0.5], [i - 0.5, i + 0.5], color="black", lw=3)
            if i + 1 < rows and coarse[i, j] != coarse[i + 1, j]:
                ax.plot([j - 0.5, j + 0.5], [i + 0.5, i + 0.5], color="black", lw=3)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"sub-clusters on last week's {rows}×{cols} SOM\n"
                 "thick lines = borders of last week's clusters · number = images on the node", fontsize=9)
    plt.tight_layout()
    plt.show()


LEVEL, SUB_INFO = MERGE_LEVEL, None          # LEVEL = the level actually used (MERGE_LEVEL is your setting)
if MERGE_LEVEL == "sub-clusters":
    sub, SUB_INFO = make_subclusters(N_SUBCLUSTERS)
    if sub is None:
        print(f"⚠️  Sub-clusters are not available: {SUB_INFO}.\n"
              "    -> Section 2 starts from last week's clusters instead.")
        LEVEL, SUB_INFO = "clusters", None
    elif SUB_INFO["k_sub"] != N_SUBCLUSTERS:
        print(f"note: N_SUBCLUSTERS = {N_SUBCLUSTERS} was changed to {SUB_INFO['k_sub']} "
              f"(at least last week's {SUB_INFO['k_week8']} clusters, at most one per SOM node)")
DATA["group"] = sub if LEVEL == "sub-clusters" else DATA.som_cluster.astype(str)

GROUPS = (DATA.groupby("group").agg(parent=("som_cluster", "first"), images=("image_id", "size"))
          .reset_index().rename(columns={"group": "name"})
          .sort_values(["parent", "images", "name"], ascending=[True, False, True]).reset_index(drop=True))
GROUPS["description"] = [describe((DATA.group == g).to_numpy()) for g in GROUPS.name]
GROUP_COLORS = {}
for parent, g in GROUPS.groupby("parent"):
    for name, amount in zip(g.name, np.linspace(0, 0.6, len(g)) if len(g) > 1 else [0.0]):
        GROUP_COLORS[name] = shade(CLUSTER_COLORS[parent % 10], amount)

if LEVEL == "sub-clusters":
    print(f"{len(GROUPS)} sub-clusters with images (each inside one of last week's {K_WEEK8} clusters)")
    plot_subcluster_map(**SUB_INFO)
display(GROUPS.rename(columns={"name": "group", "parent": "in cluster"}))

In [ ]:
# 1.5  What do the sub-clusters look like? One row per sub-cluster, the most typical image on the left
if LEVEL != "sub-clusters":
    print("No sub-clusters: Section 2 uses last week's clusters, shown in 1.3.")
elif IMG_DIR is None:
    print("No images found -> gallery skipped.")
else:
    rows = [(f"{g.name}  (part of cluster {g.parent}) · {g.images} images · {g.description}",
             example_paths(np.where(DATA.group == g.name)[0])) for g in GROUPS.itertuples()]
    gallery_rows(rows, suptitle=f"{SOURCE}: {len(GROUPS)} sub-clusters of last week's SOM")

In [ ]:
# 1.6  Profiles: how each group differs from the average image (left), and where its images come from (right)
def plot_profiles(col, order, what):
    prof = DATA.groupby(col)[READABLE].mean().reindex(order)
    z = (prof - DATA[READABLE].mean()) / DATA[READABLE].std()
    cities = [c for c in CITY_ORDER if c in set(DATA.city)] + sorted(set(DATA.city) - set(CITY_ORDER))
    comp = pd.crosstab(DATA[col], DATA.city, normalize="index").reindex(index=order, columns=cities).fillna(0)
    sizes = DATA[col].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(15, 0.42 * len(order) + 2.4), gridspec_kw={"width_ratios": [1.2, 1]})
    im = axes[0].imshow(z.to_numpy(dtype=float), cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    axes[0].set_xticks(range(len(READABLE)))
    axes[0].set_xticklabels([c[4:] for c in READABLE], rotation=45, ha="right")
    axes[0].set_yticks(range(len(order)))
    axes[0].set_yticklabels([f"{g} (n={sizes.get(g, 0)})" for g in order])
    plt.colorbar(im, ax=axes[0], fraction=0.04, label="z-score vs all images")
    axes[0].set_title(f"{what}: readable indicators (red = more than the average image)")
    values = comp.to_numpy(dtype=float)
    axes[1].imshow(values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    axes[1].set_xticks(range(len(cities)))
    axes[1].set_xticklabels(cities, rotation=45, ha="right")
    axes[1].set_yticks(range(len(order)))
    axes[1].set_yticklabels(order)
    for (r, c), v in np.ndenumerate(values):
        if v >= 0.005:
            axes[1].text(c, r, f"{v:.0%}", ha="center", va="center", fontsize=7, color="white" if v > 0.5 else "black")
    axes[1].set_title(f"{what}: which cities the images come from (each row = 100 %)")
    plt.tight_layout()
    plt.show()


plot_profiles("group", list(GROUPS.name), "sub-clusters" if LEVEL == "sub-clusters" else "last week's clusters")

✏️ **Discuss (2 min):** Which groups show the same *kind of place* to you? Is there a group that mixes very different pictures? Does a group come almost entirely from one city — and if so, is that a real urban difference, or something about the photos themselves (season, light, image quality)?

---
# 2 · Label: from groups to classes
A label is a **decision**, not a fact hidden in the data. In the panel below every group has a **class name**:

* groups with the **same name** are merged into one class;
* type **`exclude`** to leave a group out (e.g. a group that mixes everything, or one that only reflects a technical artefact);
* names are free — prefer names that describe the place, e.g. `green street`, `open & sunny`, `urban facade`.

The names start as last week's clusters (`cluster 0`, `cluster 1`, …): if you change nothing, the classes are exactly last week's clusters.
**Preview classes** shows what you would get (sizes and example pictures), **Save plan** stores it in your Drive. Then run **2.3**, which writes the new `label` column.

*Good classes:* each looks coherent in the pictures, the difference between them means something for architecture or urban design, and no class is tiny (aim for at least 30 images; 2 to 6 classes is a good start).

In [ ]:
#@title 2.1a The labelling panel — run this cell (reading its code is optional) { display-mode: "form" }
import base64
import html

EXCLUDE = "exclude"


def plan_file():
    return LABEL_DIR / f"{SOURCE}_label_plan.json"


def default_plan():
    """Every group keeps the name of the Week 8 cluster that contains it = last week's clusters."""
    return {g.name: f"cluster {g.parent}" for g in GROUPS.itertuples()}


def plan_signature():
    """The groups a plan belongs to (a saved plan is only re-used for exactly the same groups)."""
    return dict(source=SOURCE, merge_level=LEVEL, groups=list(GROUPS.name),
                sizes=[int(n) for n in GROUPS.images], week8_folder=str(WEEK8_DIR))


def load_saved_plan():
    if not plan_file().exists():
        return None
    try:
        saved = json.loads(plan_file().read_text())
    except json.JSONDecodeError:
        return None
    return saved.get("plan") if all(saved.get(k) == v for k, v in plan_signature().items()) else None


def save_plan(plan):
    plan_file().write_text(json.dumps({**plan_signature(), "plan": plan,
                                     "saved": datetime.now().isoformat(timespec="seconds")}, indent=1))


def clean_plan(plan, verbose=True):
    """Complete and tidy a plan {group: class name}: missing groups keep last week's cluster name."""
    default = default_plan()
    plan = {str(k): v for k, v in (plan or {}).items()}
    unknown = [g for g in plan if g not in default]
    missing = [g for g in default if g not in plan]
    if verbose and unknown:
        print(f"⚠️  not a group of this data, ignored: {unknown}")
    if verbose and missing:
        print(f"⚠️  not in the plan, these keep last week's cluster name: {missing}")
    out, spelling = {}, {}
    for g in default:
        name = " ".join(str(plan.get(g, default[g])).split()) or g     # tidy spaces; an empty box keeps the group name
        key = name.lower()
        if key in (EXCLUDE, "excluded"):
            out[g] = EXCLUDE
        else:                       # same name in other capitals = same class (the first spelling is kept)
            out[g] = spelling.setdefault(key, name)
    return out


def classes_of(plan):
    """{class name: [groups]} in the order of the groups (excluded groups left out)."""
    out = {}
    for g in GROUPS.name:
        if plan[g] != EXCLUDE:
            out.setdefault(plan[g], []).append(g)
    return out


def plan_report(plan):
    """Table of the classes a plan gives, number of excluded images, warnings."""
    sizes = GROUPS.set_index("name").images
    table = pd.DataFrame([{"class": c, "groups": ", ".join(gs), "images": int(sizes[gs].sum())}
                          for c, gs in classes_of(plan).items()], columns=["class", "groups", "images"])
    table["share"] = (table.images / max(table.images.sum(), 1)).map("{:.0%}".format)
    n_excluded = int(sizes[[g for g in GROUPS.name if plan[g] == EXCLUDE]].sum())
    warns = []
    if len(table) < 2:
        warns.append("only one class: a classifier needs at least two (give some groups another name)")
    small = table.loc[table.images < 30, "class"].tolist()
    if small:
        warns.append(f"small classes (fewer than 30 images: hard to learn and to test): {small}")
    if len(table) >= 2 and table.images.max() > 10 * table.images.min():
        warns.append("very unequal classes (the largest is more than 10 times the smallest)")
    return table, n_excluded, warns


def strip_jpeg(paths, height=80, gap=4):
    """A few pictures side by side, as JPEG bytes (for the panel)."""
    thumbs = []
    for p in paths:
        try:
            im = load_rgb(p)
        except Exception:
            continue
        im.thumbnail((int(height * 1.5), height))
        thumbs.append(im)
    if not thumbs:
        thumbs = [Image.new("RGB", (height, height), (230, 230, 230))]
    canvas = Image.new("RGB", (sum(t.width for t in thumbs) + gap * (len(thumbs) - 1), height), "white")
    x = 0
    for t in thumbs:
        canvas.paste(t, (x, (height - t.height) // 2))
        x += t.width + gap
    buf = io.BytesIO()
    canvas.save(buf, "JPEG", quality=85)
    return buf.getvalue()


def to_html_img(jpeg_bytes):
    return f"<img src='data:image/jpeg;base64,{base64.b64encode(jpeg_bytes).decode()}'>"


class LabelPanel:
    """One row per group: example pictures + a class name. Same name = same class; 'exclude' = left out.
    Built only from basic widgets (Text, Button, HTML, Image), which every notebook front-end can show."""

    def __init__(self, plan, n_pictures=5):
        self.signature = plan_signature()
        self.boxes, rows = {}, []
        for g in GROUPS.itertuples():
            box = W.Text(value=plan[g.name], placeholder="class name, or exclude", layout=W.Layout(width="230px"))
            box.observe(self._show_classes, names="value")
            self.boxes[g.name] = box
            head = W.HTML(f"<span style='color:{GROUP_COLORS[g.name]};font-size:18px'>&#9632;</span> "
                          f"<b>{g.name}</b> · {g.images} images<br>"
                          f"<span style='font-size:11px;color:#555'>{g.description}</span>")
            pics = example_paths(np.where(DATA.group == g.name)[0], n_pictures) if IMG_DIR is not None else []
            pictures = W.Image(value=strip_jpeg(pics), format="jpeg") if pics else W.HTML("<i>(no pictures)</i>")
            rows.append(W.HBox([W.VBox([head, box], layout=W.Layout(width="280px")), pictures],
                               layout=W.Layout(align_items="center", margin="0 0 8px 0")))
        self.classes = W.HTML()          # live summary: the classes the current names give
        self.out = W.HTML()              # preview and messages
        self.buttons = {"preview": self._button("Preview classes", "info", self.preview),
                        "save": self._button("Save plan", "success", self.save),
                        "reset": self._button("Reset: last week's clusters", "", lambda: self.set_plan(default_plan()))}
        if LEVEL == "sub-clusters":
            self.buttons["split"] = self._button("Every sub-cluster its own class", "",
                                                 lambda: self.set_plan({g: g for g in GROUPS.name}))
        self.widget = W.VBox(rows + [self.classes, W.HBox(list(self.buttons.values())), self.out])
        self._show_classes()

    def _button(self, text, style, action):
        b = W.Button(description=text, button_style=style, layout=W.Layout(width="auto"))
        b.on_click(lambda _: action())
        return b

    def _show_classes(self, change=None):
        plan = self.plan()
        parts = [f"<b>{html.escape(c)}</b> <span style='color:#777'>({', '.join(gs)})</span>"
                 for c, gs in classes_of(plan).items()]
        excluded = [g for g in GROUPS.name if plan[g] == EXCLUDE]
        self.classes.value = (f"<div style='margin:6px 0'>classes now: {' · '.join(parts) or '–'}"
                              + (f" · <i>excluded: {', '.join(excluded)}</i>" if excluded else "") + "</div>")

    def plan(self):
        return clean_plan({g: b.value for g, b in self.boxes.items()}, verbose=False)

    def set_plan(self, plan):
        for g, b in self.boxes.items():
            b.value = plan[g]
        self.preview()

    def preview(self):
        plan = self.plan()
        table, n_excluded, warns = plan_report(plan)
        rng = np.random.default_rng(SEED)
        parts = [f"<p><b>{len(table)} classes</b> from {len(GROUPS)} groups"
                 + (f" · {n_excluded} images excluded" if n_excluded else "") + "</p>",
                 table.to_html(index=False, escape=True)]
        parts += [f"<p style='color:#c0392b'>⚠️ {html.escape(w)}</p>" for w in warns]
        if IMG_DIR is not None:
            for c, gs in classes_of(plan).items():
                idx = np.where(DATA.group.isin(gs))[0]
                pick = rng.choice(idx, size=min(8, len(idx)), replace=False)
                parts.append(f"<p style='margin:8px 0 2px'><b>{html.escape(c)}</b> "
                             f"<span style='color:#777'>({', '.join(gs)})</span></p>"
                             + to_html_img(strip_jpeg([IMG_DIR / p for p in DATA.relpath.iloc[pick]], height=70)))
        self.out.value = "".join(parts)

    def save(self):
        plan = self.plan()
        save_plan(plan)
        self.out.value = (f"<p>✅ Saved to <code>{html.escape(str(plan_file()))}</code>. Now run cell <b>2.3</b>.</p>"
                          f"<p>The same plan as code (for <code>MANUAL_PLAN</code> in 2.2):</p>"
                          f"<pre>MANUAL_PLAN = {html.escape(repr(plan))}</pre>")

In [ ]:
# 2.1b  The panel: it starts from your saved plan if there is one, otherwise from last week's clusters
saved = load_saved_plan()
PANEL = LabelPanel(clean_plan(saved, verbose=False) if saved else default_plan())
print("Starting from your saved plan." if saved else "Starting from last week's clusters (no saved plan yet).")
print("Change the class names  ->  'Preview classes'  ->  'Save plan'  ->  run cell 2.3")
display(PANEL.widget)

In [ ]:
# 2.2  (Optional) The same without the panel: write the plan as a dictionary {group: class name}.
#      Leave MANUAL_PLAN = None to use the panel. "Save plan" prints your current plan in this format.
MANUAL_PLAN = None
# Example for street view with 8 sub-clusters (use the group names YOU see in the panel):
# MANUAL_PLAN = {"0a": "street trees", "0b": "tree canopy", "0c": "lawn & house", "0d": "tree canopy",
#                "1a": "open space", "1b": "low-rise & parking", "1c": "lawn & house", "2a": "urban facade"}

In [ ]:
# 2.3  Apply the plan: new columns `label` (class name) and `label_id` (class number), saved as a CSV in Week9/labels/
def current_plan():
    if MANUAL_PLAN:
        return clean_plan(MANUAL_PLAN), "MANUAL_PLAN (cell 2.2)"
    if "PANEL" in globals() and PANEL.signature == plan_signature():
        return PANEL.plan(), "the panel (2.1)"
    if "PANEL" in globals():
        print("⚠️  The panel was made for other groups (a setting changed since 2.1b ran) and is ignored.\n"
              "    Run 2.1a and 2.1b again, choose your names, then run this cell again.")
    saved = load_saved_plan()
    if saved:
        return clean_plan(saved), f"your saved plan ({plan_file().name})"
    return default_plan(), "last week's clusters (default)"


PLAN, plan_origin = current_plan()
CLASS_TABLE, n_excluded, warns = plan_report(PLAN)
CLASSES = list(classes_of(PLAN))
if len(CLASSES) < 2:
    raise ValueError("Only one class: a classifier needs at least two. Give some groups a different name (2.1 or 2.2).")
too_small = CLASS_TABLE.loc[CLASS_TABLE.images < 5, "class"].tolist()
if too_small:
    raise ValueError(f"Classes with fewer than 5 images cannot be split into train / validation / test: {too_small}. "
                     "Merge them with another class or exclude them.")
N_CLASSES = len(CLASSES)
named = DATA.group.map(PLAN)
DATA["label"] = named.where(named != EXCLUDE)
DATA["label_id"] = DATA.label.map({c: i for i, c in enumerate(CLASSES)}).astype("Int64")
LABELS_STAMP = time.time()          # lets Section 3 notice when the labels change
save_plan(PLAN)
LABELLED_CSV = LABEL_DIR / f"{SOURCE}_labelled.csv"


def save_labelled_csv():
    out = DATA.drop(columns=["group"])
    if LEVEL == "sub-clusters":
        out.insert(out.columns.get_loc("label"), "subcluster", DATA.group)
    out.to_csv(LABELLED_CSV, index=False, float_format="%.8g")


save_labelled_csv()
print(f"plan taken from {plan_origin}")
display(CLASS_TABLE.assign(label_id=range(N_CLASSES))[["label_id", "class", "groups", "images", "share"]])
for w in warns:
    print("⚠️ ", w)
if n_excluded:
    print(f"{n_excluded} images are excluded (their label is empty).")
print(f"-> {LABELLED_CSV}  ({len(DATA)} rows, new columns: "
      f"{'subcluster, ' if LEVEL == 'sub-clusters' else ''}label, label_id)")

In [ ]:
# 2.4  Your classes: random examples, profiles and cities
if IMG_DIR is not None:
    rng = np.random.default_rng(SEED)
    rows = []
    for c in CLASSES:
        idx = np.where(DATA.label == c)[0]
        pick = rng.choice(idx, size=min(8, len(idx)), replace=False)
        rows.append((f"{c} · {len(idx)} images", [IMG_DIR / p for p in DATA.relpath.iloc[pick]]))
    gallery_rows(rows, suptitle="your classes — random examples")
plot_profiles("label", CLASSES, "your classes")

---
# 3 · Learn: supervised models

| | Unsupervised (Weeks 5–8) | Supervised (today) |
|---|---|---|
| we give the algorithm | the numbers of each image (**X**) | the numbers **X** *and* a label **y** for each image |
| it learns | groups of similar images | a rule **X → y** |
| we use it to | explore and describe | predict the label of images it has **never seen** |

**A fair test.** A model can simply memorise the images it learnt from, so we judge it on images it has not seen. The labelled images are split in three:
* **train** (≈ 64 %) — the model learns from these (the *homework*);
* **validation** (≈ 16 %) — used to make choices: tree depth, when to stop training, which model is best (the *mock exam*);
* **test** (20 %) — looked at only at the end, to report how good the chosen model is (the *final exam*).

**Scores.** *Accuracy* = share of images that get the right label. With unequal classes it misleads: if 56 % of the images are "green street", always answering "green street" already gives 56 %. **Balanced accuracy** averages the accuracy *within each class* (always answering the same class gives 1 / number of classes). The **confusion matrix** shows which classes get mixed up. Until the final exam (3.4c) every model is judged on the **validation** images only.

> ⚠️ **Be honest about what we test.** Your labels come from the SOM, and the SOM was trained on the `emb_` columns. A model that predicts the labels from `emb_` learns to **imitate the SOM**: it should do well, and that is no discovery (it is still useful — the model is fast, gives a confidence, and shows how clearly your classes are separated).
> The readable `int_` indicators are not an independent test either: they are computed from the same lenses (segmentation, edges, colour) that make up most of the embedding. What they add is **readability** — they show *which* understandable properties separate your classes (3.2).
> Two questions are more independent: can the model classify a **city it has never seen** (`SPLIT = "by city"` in 3.1, exercise 2)? Can **another view of the same place** predict the class — the satellite image predicting the street-view class (3.5)?
> *(Strictly speaking, last week's PCA and SOM saw all images — today's test images too — so every score here is a little optimistic.)*

In [ ]:
#@title 3.1 Split the labelled images: train / validation / test
SPLIT = "random"          #@param ["random", "by city"]
TEST_CITY = "Miami"       #@param ["Gainesville", "Tallahassee", "Jacksonville", "Orlando", "Tampa", "Miami", "Saint-Malo"]
TEST_SHARE = 0.2          # SPLIT = "random": share of the images kept for the final test
VAL_SHARE = 0.2           # share of the other images used for validation

from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore", message=".*y_pred contains classes not in y_true.*")

LAB = DATA[DATA.label.notna()].reset_index(drop=True)          # the labelled images
Y = LAB.label_id.to_numpy(dtype=int)                           # their class numbers


def two_way(positions, share):
    """Split positions in two, keeping the class proportions when possible."""
    try:
        return train_test_split(positions, test_size=share, stratify=Y[positions], random_state=SEED)
    except ValueError:                                         # a class too small to keep its proportion
        return train_test_split(positions, test_size=share, random_state=SEED)


positions = np.arange(len(LAB))
if SPLIT == "by city":
    if TEST_CITY not in set(LAB.city):
        raise ValueError(f"TEST_CITY = {TEST_CITY!r} is not in the {SOURCE} data. Choose one of {sorted(set(LAB.city))}.")
    in_city = (LAB.city == TEST_CITY).to_numpy()
    rest, TEST = positions[~in_city], positions[in_city]
else:
    rest, TEST = two_way(positions, TEST_SHARE)
TRAIN, VAL = two_way(rest, VAL_SHARE)
TRAIN, VAL, TEST = np.sort(TRAIN), np.sort(VAL), np.sort(TEST)
LAB["split"] = "train"
LAB.loc[VAL, "split"] = "validation"
LAB.loc[TEST, "split"] = "test"
SPLIT_STAMP, SPLIT_ID = LABELS_STAMP, time.time()             # lets later cells notice a new labelling or split
RESULTS, MODELS = [], {}                                        # every trained model is added here (scoreboard 3.4)

counts = (pd.crosstab(LAB.label, LAB.split).reindex(index=CLASSES, columns=["train", "validation", "test"])
          .fillna(0).astype(int))
counts.loc["all classes"] = counts.sum()
display(counts)
if SPLIT == "by city":
    print(f"test = all images of {TEST_CITY}: the models never see this city while learning")
    absent = [c for c in CLASSES if counts.loc[c, "test"] == 0]
    if absent:
        print(f"⚠️  {TEST_CITY} has no images of {absent}. The test *balanced* accuracy only averages the classes that "
              "are present, so here compare the test *accuracy* and the confusion matrix (3.4c).")
missing = [c for c in CLASSES if counts.loc[c, "train"] == 0]
if missing:
    print(f"⚠️  no training images for {missing}: no model can learn these classes")

In [ ]:
# 3.1b  Tools used by every model: feature sets, preprocessing, scoreboard
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

FEATURE_SETS = {"int": INT_COLS, "emb": EMB_COLS, "emb+int": EMB_COLS + INT_COLS}


def make_preprocessor(columns, scale=True):
    """int_ columns have different units -> standardised (mean 0, std 1), unless the model does not need it.
    emb_ columns are PCA coordinates on one common scale -> passed on unchanged (3.2g shows why)."""
    ints = [c for c in columns if not c.startswith("emb_")]
    embs = [c for c in columns if c.startswith("emb_")]
    parts = []
    if ints:
        parts.append(("int", StandardScaler() if scale else "passthrough", ints))
    if embs:
        parts.append(("emb", "passthrough", embs))
    return ColumnTransformer(parts)


def check_labels_unchanged():
    if SPLIT_STAMP != LABELS_STAMP:
        raise RuntimeError("The labels changed (cell 2.3 was run again): re-run 3.1 first, then the cells below it.")


def scores(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred), "bal_acc": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


def record(model_name, features, predictor, X_val, X_test, seconds, kind="this view", val_idx=None, test_idx=None,
           quiet=False):
    """Score a trained model and add it to the scoreboard. Only the validation score is shown now; the test
    score is kept hidden until the final exam (3.4c)."""
    val_idx = VAL if val_idx is None else val_idx
    test_idx = TEST if test_idx is None else test_idx
    row = {"model": model_name, "features": features, "kind": kind, "seconds": round(seconds, 1)}
    for part, X_part, idx in (("val", X_val, val_idx), ("test", X_test, test_idx)):
        row.update({f"{part}_{k}": v for k, v in scores(Y[idx], predictor.predict(X_part)).items()})
    RESULTS[:] = [r for r in RESULTS if (r["model"], r["features"]) != (model_name, features)]
    RESULTS.append(row)
    MODELS[(model_name, features)] = predictor
    if not quiet:
        print(f"{model_name} [{features}]  validation images: accuracy {row['val_acc']:.2f}, "
              f"balanced accuracy {row['val_bal_acc']:.2f}")
    return row


def fit_and_record(model_name, estimator, features, scale=True, columns=None, data=None, kind="this view",
                   idx=None, features_label=None, quiet=False):
    """The whole recipe of supervised learning: preprocess + learn from the training images, then score."""
    check_labels_unchanged()
    columns = columns or FEATURE_SETS[features]
    data = LAB if data is None else data
    tr, va, te = idx or (TRAIN, VAL, TEST)
    model = make_pipeline(make_preprocessor(columns, scale), estimator)
    t0 = time.time()
    model.fit(data.loc[tr, columns], Y[tr])
    record(model_name, features_label or features, model, data.loc[va, columns], data.loc[te, columns],
           time.time() - t0, kind, va, te, quiet)
    return model


MODEL_ORDER = ["majority class (baseline)", "decision tree (depth 3)", "logistic regression", "k-NN (k = 10)",
               "random forest", "gradient boosting", "MLP (PyTorch)"]


def results_table(test=False):
    """The scoreboard, best validation score first. Test scores only with test=True (after the choice is made)."""
    t = pd.DataFrame(RESULTS)
    t["order"] = [MODEL_ORDER.index(m) if m in MODEL_ORDER else len(MODEL_ORDER) for m in t.model]
    t = t.sort_values(["val_bal_acc", "order"], ascending=[False, True])
    cols = ["model", "features", "kind", "val_acc", "val_bal_acc"] + (["test_acc", "test_bal_acc"] if test else [])
    return t[cols + ["seconds"]].reset_index(drop=True).round(3)


def best_choice():
    """The model with the best validation score, among the models on this view's CSV columns."""
    board = results_table()
    return board[(board.kind == "this view") & (board.features != "–")].iloc[0]


def proba_all(model, X):
    """Class probabilities with one column per class (also if a class had no training images)."""
    P = np.asarray(model.predict_proba(X))
    classes = np.asarray(getattr(model, "classes_", np.arange(P.shape[1])), dtype=int)
    out = np.zeros((P.shape[0], N_CLASSES))
    out[:, classes] = P
    return out


print("feature sets:", {k: f"{len(v)} columns" for k, v in FEATURE_SETS.items()})

## 3.2 Models you can read
**(a) Baseline.** Always answer the most common class. A model is only useful if it beats this.

In [ ]:
# 3.2a  Baseline
from sklearn.dummy import DummyClassifier
fit_and_record("majority class (baseline)", DummyClassifier(strategy="most_frequent"), "int", features_label="–");

**(b) Decision tree.** A sequence of yes/no questions on the readable indicators ("is the vegetation share above 0.31?"); each question splits the images so that the groups become as pure as possible. The deeper the tree, the more questions in a row — and the more it can **memorise** the training images instead of learning a general rule: **overfitting**.
✏️ Move the slider and compare the two curves: the score on the training images keeps rising with depth, the score on the validation images rises much less or stops. The gap between the curves is overfitting.
*Reading a box of the tree:* the question, the share of the training images that reach the box, and the share of each class among them — classes are weighted equally (so a small class is not ignored), which is why at the top every class has the same share. The colour shows the class that wins in the box.

In [ ]:
# 3.2b  Decision tree on the readable indicators — move the slider
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

check_labels_unchanged()
INT_NAMES = [c[4:] for c in INT_COLS]                     # "int_vegetation" -> "vegetation"
X_tr_int, X_va_int = LAB.loc[TRAIN, INT_COLS], LAB.loc[VAL, INT_COLS]
DEPTHS = list(range(1, 13))
depth_scores = []
for d in DEPTHS:
    t = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=SEED).fit(X_tr_int, Y[TRAIN])
    depth_scores.append((balanced_accuracy_score(Y[TRAIN], t.predict(X_tr_int)),
                         balanced_accuracy_score(Y[VAL], t.predict(X_va_int))))
depth_scores = np.array(depth_scores)


def tree_figure(max_depth=3):
    """Left: training / validation score for every depth. Right: the tree with the chosen depth."""
    tree = DecisionTreeClassifier(max_depth=max_depth, class_weight="balanced", random_state=SEED).fit(X_tr_int, Y[TRAIN])
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 3]})
    axes[0].plot(DEPTHS, depth_scores[:, 0], "o-", label="training images (homework)")
    axes[0].plot(DEPTHS, depth_scores[:, 1], "s-", label="validation images (mock exam)")
    axes[0].axvline(max_depth, color="crimson", ls="--", lw=1)
    axes[0].set(xlabel="max_depth (questions in a row)", ylabel="balanced accuracy", ylim=(0, 1.02))
    axes[0].legend(fontsize=8, loc="lower right")
    axes[0].set_title("score for every depth")
    if max_depth <= 3:
        plot_tree(tree, feature_names=INT_NAMES, class_names=[CLASSES[c] for c in tree.classes_], filled=True,
                  impurity=False, proportion=True, rounded=True, label="root", fontsize=8, ax=axes[1])
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, f"{tree.get_n_leaves()} leaves = {tree.get_n_leaves()} rules: too big to draw.\n"
                     "Compare the two curves on the left.", ha="center", va="center", fontsize=12)
    axes[1].set_title(f"the tree with max_depth = {max_depth}")
    fig.tight_layout()
    return fig


def draw_tree(change=None):
    fig = tree_figure(TREE_SLIDER.value)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    TREE_VIEW.value = buf.getvalue()


TREE_SLIDER = W.IntSlider(value=3, min=1, max=12, continuous_update=False, description="max_depth")
TREE_VIEW = W.Image(format="png", layout=W.Layout(width="100%"))
TREE_SLIDER.observe(draw_tree, names="value")
draw_tree()
display(W.VBox([TREE_SLIDER, TREE_VIEW]))

**(c) The tree as rules.** For the comparison we keep a tree with 3 questions in a row — chosen because it is still readable, not because it scores best. Each line below is a question; the class at the end of a branch is the answer. Sometimes both branches of a question end in the same class: the question still made the groups purer, it just does not change the answer.

In [ ]:
# 3.2c  Keep the depth-3 tree for the comparison, and read it as rules
TREE = fit_and_record("decision tree (depth 3)",
                      DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED), "int", scale=False)
rules = export_text(TREE[-1], feature_names=INT_NAMES, decimals=3)
print(re.sub(r"class: (\d+)", lambda m: "class: " + CLASSES[int(m.group(1))], rules))

**(d) Logistic regression.** Every indicator gets a **weight** for every class; an image's score for a class is the weighted sum of its (standardised) indicators, and the scores are turned into probabilities. Large positive weight = "more of this makes the class more likely".

In [ ]:
# 3.2d  Logistic regression on the readable indicators: what pushes an image towards each class?
from sklearn.linear_model import LogisticRegression
LOGREG = fit_and_record("logistic regression", LogisticRegression(max_iter=3000, class_weight="balanced"), "int")
lr_model = LOGREG[-1]
coef = lr_model.coef_
row_names = ([CLASSES[c] for c in lr_model.classes_] if coef.shape[0] > 1 else
             [f"{CLASSES[lr_model.classes_[1]]}  (vs {CLASSES[lr_model.classes_[0]]})"])
lim = np.abs(coef).max()
fig, ax = plt.subplots(figsize=(0.5 * len(INT_COLS) + 3, 0.5 * len(row_names) + 2.4))
im = ax.imshow(coef, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(INT_NAMES)))
ax.set_xticklabels(INT_NAMES, rotation=50, ha="right")
ax.set_yticks(range(len(row_names)))
ax.set_yticklabels(row_names)
plt.colorbar(im, ax=ax, fraction=0.03, label="weight")
ax.set_title("logistic regression — red: more of this indicator makes the class more likely, blue: less likely")
plt.tight_layout()
plt.show()

**(e) Random forest.** Many decision trees (here 200) that vote. Each tree learns from a random sample of the training images, and at every question it may only choose among a random few of the indicators — so the trees differ, and their mistakes partly cancel out. Usually much better than one tree, but no longer readable as rules. To see what the forest relies on, we **shuffle one indicator at a time** in the validation images and measure how much the score drops (*permutation importance*).

In [ ]:
# 3.2e  Random forest + which indicators matter
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
FOREST = fit_and_record("random forest", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                                random_state=SEED, n_jobs=-1), "int", scale=False)
imp = permutation_importance(FOREST, LAB.loc[VAL, INT_COLS], Y[VAL], scoring="balanced_accuracy", n_repeats=10,
                             random_state=SEED, n_jobs=-1)
top = np.argsort(imp.importances_mean)[::-1][:12][::-1]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh([INT_NAMES[i] for i in top], imp.importances_mean[top], xerr=imp.importances_std[top], color="#4c72b0")
ax.set_xlabel("drop of the validation balanced accuracy when this indicator is shuffled")
ax.set_title("random forest: which indicators does it rely on?")
plt.tight_layout()
plt.show()

**(f) k-nearest neighbours (k-NN).** No training at all: to classify an image, find the *k* most similar training images in the embedding and let them vote. It is the "similar images" search of Week 8 (2.10a), used for prediction.

In [ ]:
# 3.2f  k-NN on the embedding
from sklearn.neighbors import KNeighborsClassifier
KNN = fit_and_record("k-NN (k = 10)", KNeighborsClassifier(n_neighbors=10), "emb")

In [ ]:
# 3.2g  (optional) A lesson about preprocessing: the same k-NN after standardising every emb_ column
knn_std = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10)).fit(LAB.loc[TRAIN, EMB_COLS], Y[TRAIN])
print(f"k-NN on the embedding as it is:        validation balanced accuracy "
      f"{balanced_accuracy_score(Y[VAL], KNN.predict(LAB.loc[VAL, EMB_COLS])):.2f}")
print(f"k-NN after standardising every column: validation balanced accuracy "
      f"{balanced_accuracy_score(Y[VAL], knn_std.predict(LAB.loc[VAL, EMB_COLS])):.2f}   <- why?")
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.bar(range(len(EMB_COLS)), LAB[EMB_COLS].std().to_numpy(), width=1.0, color="#4c72b0")
ax.set(xlabel=f"embedding column ({EMB_COLS[0]} … {EMB_COLS[-1]})", ylabel="standard deviation",
       title="spread of the embedding columns: the first PCA directions carry most of the differences")
plt.tight_layout()
plt.show()

**Why did standardising hurt?** The SOM that made your labels compared images with the embedding *as it is*: there, the first PCA directions have by far the largest spread and decided almost everything, while the many later directions hardly mattered. Standardising gives every column the same spread, so the directions that played almost no role in your labels suddenly count as much as the important ones — and the nearest neighbours change. Preprocessing is not a ritual: it must fit what the numbers mean, and here also how the labels were made. That is why, in this notebook, `int_` columns are standardised and `emb_` columns are not.

In [ ]:
# 3.2h  (optional) Why does k-NN decide what it decides? A validation image and its 5 nearest training images
QUERY = 0        # ✏️ try other numbers
q = VAL[QUERY % len(VAL)]
emb_train = LAB.loc[TRAIN, EMB_COLS].to_numpy(dtype=float)
dist = np.linalg.norm(emb_train - LAB.loc[q, EMB_COLS].to_numpy(dtype=float), axis=1)
nearest = np.argsort(dist)[:5]
said = CLASSES[int(KNN.predict(LAB.loc[[q], EMB_COLS])[0])]
if IMG_DIR is None:
    print("No images found.")
else:
    show_images([IMG_DIR / LAB.relpath[i] for i in [q, *TRAIN[nearest]]],
                [f"VALIDATION IMAGE\nlabel: {LAB.label[q]}\nk-NN says: {said}"]
                + [f"neighbour {j + 1}: {LAB.label[TRAIN[n]]}\ndistance {dist[n]:.2f}" for j, n in enumerate(nearest)],
                ncols=6, size=2.6)

## 3.3 Deep learning: a neural network (multi-layer perceptron, MLP)
Each **neuron** takes all numbers of the previous layer, multiplies them by its weights, adds them up and passes the sum through a simple non-linear function (ReLU: negative → 0). Stacking layers lets the network build more and more complex combinations of the inputs.

```
 input: the emb_ numbers      hidden layer 1       hidden layer 2       output: one score per class
   emb_000 ──┐                ┌── neuron ──┐       ┌── neuron ──┐       ┌── "green street"   72 %
   emb_001 ──┤   weights      ├── neuron ──┤       ├── neuron ──┤       ├── "open & sunny"   21 %
     …       ├── + ReLU ────► │    … 64    ├─────► │    … 32    ├─────► └── "urban facade"    7 %
   emb_…   ──┘                └── neuron ──┘       └── neuron ──┘         (softmax: scores → probabilities)
```

**Training** = finding good weights, by repeating:
1. take a small **batch** of 32 training images and let the network guess their classes;
2. measure how wrong the guesses are — the **loss** (cross-entropy: large when the network is confident *and* wrong; rare classes count more, so they are not ignored);
3. compute in which direction every weight should move to reduce the loss (**backpropagation**) and move it a little (the **optimizer** Adam; the step size is the **learning rate**).

One pass over all training images is an **epoch**. After every epoch the network is checked on the **validation** images; training stops when the validation loss has not improved for `PATIENCE` epochs (**early stopping**), and the best epoch is kept.
Two more defences against overfitting: **dropout** (switch off a random 20 % of the neurons at every step, so the network cannot rely on single neurons) and **weight decay** (keep the weights small).

In [ ]:
#@title 3.3a Settings and the network
import torch
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MLP_FEATURES = "emb"      #@param ["emb", "int", "emb+int"]
HIDDEN = (64, 32)         # neurons in each hidden layer
DROPOUT = 0.2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32
EPOCHS = 300              # at most
PATIENCE = 30             # early stopping: stop after this many epochs without improvement (None = never stop early)


def make_mlp(n_inputs, n_outputs, hidden=HIDDEN, dropout=DROPOUT):
    layers, width = [], n_inputs
    for h in hidden:
        layers += [nn.Linear(width, h), nn.ReLU(), nn.Dropout(dropout)]
        width = h
    layers.append(nn.Linear(width, n_outputs))   # one score per class (softmax turns the scores into probabilities)
    return nn.Sequential(*layers)


torch.manual_seed(SEED)
example = make_mlp(len(FEATURE_SETS[MLP_FEATURES]), N_CLASSES)
layout = " -> ".join(str(n) for n in [len(FEATURE_SETS[MLP_FEATURES]), *HIDDEN, N_CLASSES])
print(f"network: {layout} neurons | {sum(p.numel() for p in example.parameters()):,} weights to learn "
      f"from {len(TRAIN)} training images (computing on {DEVICE.type.upper()})")

In [ ]:
# 3.3b  The training loop (the same function trains the tiny CNN of 3.6), then train the MLP
def to_tensor(x, dtype=torch.float32):
    return torch.as_tensor(np.asarray(x), dtype=dtype)


@torch.no_grad()
def predict_scores(model, X, batch_size=256):
    model.eval()
    return torch.cat([model(xb.to(DEVICE)).cpu() for xb in X.split(batch_size)])


def train_network(model, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                  batch_size=BATCH_SIZE, patience=PATIENCE, augment=None, keep_best=True, desc="training"):
    """Mini-batch training with Adam. Returns the model (at its best validation epoch), the history, the best epoch."""
    torch.manual_seed(SEED)
    model = model.to(DEVICE)
    Xtr, ytr = to_tensor(X_tr), to_tensor(y_tr, torch.long)
    Xva, yva = to_tensor(X_va), to_tensor(y_va, torch.long)
    counts = np.bincount(np.asarray(y_tr), minlength=N_CLASSES)
    class_weight = torch.tensor(len(y_tr) / (N_CLASSES * np.maximum(counts, 1)), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight.to(DEVICE))      # rare classes count more
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    order = torch.Generator().manual_seed(SEED)
    history, best_loss, best_state, best_epoch = [], np.inf, None, 0
    for epoch in tqdm(range(1, epochs + 1), desc=desc, leave=False):
        model.train()
        total_loss = 0.0
        for batch in torch.randperm(len(Xtr), generator=order).split(batch_size):
            xb, yb = Xtr[batch].to(DEVICE), ytr[batch].to(DEVICE)
            if augment is not None:
                xb = augment(xb)
            optimizer.zero_grad()
            out = model(xb)                       # 1. guess
            loss = loss_fn(out, yb)               # 2. how wrong?
            loss.backward()                       # 3. backpropagation …
            optimizer.step()                      #    … and a small step
            total_loss += loss.item() * len(batch)
        val_out = predict_scores(model, Xva)
        val_loss = loss_fn(val_out.to(DEVICE), yva.to(DEVICE)).item()
        history.append({"epoch": epoch, "train_loss": total_loss / len(Xtr), "val_loss": val_loss,
                        "val_bal_acc": balanced_accuracy_score(y_va, val_out.argmax(1).numpy())})
        if val_loss < best_loss:
            best_loss, best_epoch, best_state = val_loss, epoch, copy.deepcopy(model.state_dict())
        elif patience and epoch - best_epoch >= patience:
            break
    if keep_best and best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


class NetworkClassifier:
    """A trained network used like the scikit-learn models: predict() and predict_proba()."""

    def __init__(self, model, prep=None, columns=None, config=None):
        self.model, self.prep, self.columns, self.config = model, prep, columns, config or {}

    def _inputs(self, X):
        return self.prep.transform(X[self.columns]) if self.prep is not None else np.asarray(X)

    def predict_proba(self, X):
        return torch.softmax(predict_scores(self.model, to_tensor(self._inputs(X))), dim=1).numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def plot_history(history, best_epoch=None, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(history.epoch, history.train_loss, label="training images")
    axes[0].plot(history.epoch, history.val_loss, label="validation images")
    axes[0].set_title("loss (lower = better)")
    axes[1].plot(history.epoch, history.val_bal_acc, color="tab:orange", label="validation images")
    axes[1].set(title="balanced accuracy", ylim=(0, 1.02))
    for ax in axes:
        ax.set_xlabel("epoch")
        if best_epoch:
            ax.axvline(best_epoch, color="crimson", ls="--", lw=1, label=f"kept: epoch {best_epoch}")
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


def fit_mlp(features, columns=None, data=None, kind="this view", idx=None, hidden=HIDDEN, dropout=DROPOUT,
            name="MLP (PyTorch)", quiet=False, **train_kw):
    """Standardise / pass the columns (as in 3.1b), train an MLP, add it to the scoreboard."""
    check_labels_unchanged()
    columns = columns or FEATURE_SETS[features]
    data = LAB if data is None else data
    tr, va, te = idx or (TRAIN, VAL, TEST)
    prep = make_preprocessor(columns).fit(data.loc[tr, columns])
    torch.manual_seed(SEED)
    t0 = time.time()
    net, history, best = train_network(make_mlp(len(columns), N_CLASSES, hidden, dropout),
                                       prep.transform(data.loc[tr, columns]), Y[tr],
                                       prep.transform(data.loc[va, columns]), Y[va], desc=name, **train_kw)
    clf = NetworkClassifier(net, prep, columns, {"n_inputs": len(columns), "hidden": list(hidden), "dropout": dropout})
    record(name, features, clf, data.loc[va, columns], data.loc[te, columns], time.time() - t0, kind, va, te, quiet)
    return clf, history, best


MLP, MLP_HISTORY, MLP_BEST = fit_mlp(MLP_FEATURES)
plot_history(MLP_HISTORY, MLP_BEST, f"MLP {HIDDEN} on '{MLP_FEATURES}' — dropout {DROPOUT}, weight decay {WEIGHT_DECAY}")

**Reading the curves.** At first both losses fall. When the validation loss stops falling while the training loss keeps going down, the network starts to memorise the training images — we keep the epoch with the lowest validation loss (red line). (In the first epochs the training loss can be *higher* than the validation loss: during training, dropout switches neurons off; on the validation images the full network is used.)

In [ ]:
# 3.3c  (optional) Overfit on purpose: a much bigger network without dropout, weight decay or early stopping
BIG, BIG_HISTORY, _ = fit_mlp(MLP_FEATURES, hidden=(512, 512), dropout=0.0, name="MLP big, no regularisation",
                              kind="demo", weight_decay=0.0, patience=None, epochs=300, keep_best=False)
plot_history(BIG_HISTORY, None, "big network (512, 512) without regularisation: training loss -> 0, validation loss goes up")

**What happened?** The big network (about 300,000 weights for a few hundred images) drives the training loss to almost zero — it has memorised the training images — while the validation loss keeps rising: on images it has not seen it becomes more and more *over-confident*, also when it is wrong. Its accuracy may hardly change (it can even be as good as the small network): the damage is in the confidence, which can no longer be trusted. That is why we keep dropout, weight decay and early stopping. *(This demo is not on the scoreboard.)*

## 3.4 Compare all models
Every model is now trained on the three feature sets: `int` (the readable indicators), `emb` (the embedding the SOM used) and `emb+int`. *(About a minute.)* One newcomer: **gradient boosting** — small decision trees built one after another, each one correcting the mistakes of the trees before it; often the strongest model for tables of numbers.
The scoreboard shows only **validation** scores: we choose with the mock exam. For the neural networks this score is a little optimistic, because the validation images also decided when their training stopped.

In [ ]:
# 3.4a  All models × all feature sets
from sklearn.ensemble import HistGradientBoostingClassifier

ZOO = {   # name: (new model, standardise the int_ columns?)
    "k-NN (k = 10)": (lambda: KNeighborsClassifier(n_neighbors=10), True),
    "logistic regression": (lambda: LogisticRegression(max_iter=3000, class_weight="balanced"), True),
    "decision tree (depth 3)": (lambda: DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED), False),
    "random forest": (lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED,
                                                     n_jobs=-1), False),
    "gradient boosting": (lambda: HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED), False),
}
jobs = [(name, features) for features in FEATURE_SETS for name in [*ZOO, "MLP (PyTorch)"] if (name, features) not in MODELS]
for name, features in tqdm(jobs, desc="training"):
    if name == "MLP (PyTorch)":
        fit_mlp(features, quiet=True)
    else:
        new_model, scale = ZOO[name]
        fit_and_record(name, new_model(), features, scale=scale, quiet=True)
print(f"{len(jobs)} models trained -> scoreboard below")

In [ ]:
# 3.4b  The scoreboard (validation images) — the mock exam decides which model we take
BOARD = results_table()
BOARD = BOARD[BOARD.kind != "demo"].reset_index(drop=True)
display(BOARD)
pivot = BOARD[BOARD.kind == "this view"].pivot_table(index="model", columns="features", values="val_bal_acc")
pivot = pivot.reindex(index=[m for m in MODEL_ORDER if m in pivot.index] + [m for m in pivot.index if m not in MODEL_ORDER],
                      columns=[c for c in ["–", "int", "emb", "emb+int"] if c in pivot.columns])
fig, ax = plt.subplots(figsize=(1.5 * pivot.shape[1] + 4, 0.45 * pivot.shape[0] + 1.6))
ax.imshow(pivot.to_numpy(dtype=float), cmap="Greens", vmin=1 / N_CLASSES, vmax=1, aspect="auto")
for (r, c), v in np.ndenumerate(pivot.to_numpy(dtype=float)):
    if not np.isnan(v):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v > 1 / N_CLASSES + 0.6 * (1 - 1 / N_CLASSES) else "black")
ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index)
ax.set_title(f"validation balanced accuracy (chance = 1 / {N_CLASSES} = {1 / N_CLASSES:.2f})")
plt.tight_layout()
fig.savefig(FIG_DIR / f"{SOURCE}_model_comparison.png", dpi=110, bbox_inches="tight")
plt.show()

### 3.4c The final exam
The model with the best validation score now meets the **test** images — once. Besides accuracy and balanced accuracy, the table below gives two scores per class:
* **recall** — of the test images that carry this label, the share the model recognised;
* **precision** — of the test images the model put in this class, the share that really carry this label.

In [ ]:
# 3.4c  The final exam: the chosen model on the test images
from sklearn.metrics import ConfusionMatrixDisplay, precision_score, recall_score
best_row = best_choice()
BEST_KEY = (best_row.model, best_row.features)
best_model, best_cols = MODELS[BEST_KEY], FEATURE_SETS[best_row.features]
P_test = proba_all(best_model, LAB.loc[TEST, best_cols])
pred_test = P_test.argmax(axis=1)
y_test = Y[TEST]
print(f"chosen with the validation images: {best_row.model} on '{best_row.features}' "
      f"(validation balanced accuracy {best_row.val_bal_acc:.2f})")
print(f"TEST images: accuracy {accuracy_score(y_test, pred_test):.2f} | "
      f"balanced accuracy {balanced_accuracy_score(y_test, pred_test):.2f}")
absent = [CLASSES[c] for c in range(N_CLASSES) if not (y_test == c).any()]
if absent:
    print(f"⚠️  the test images contain no {absent}: balanced accuracy only averages the other classes — "
          "trust the accuracy and the confusion matrix here.")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, norm, fmt, title in ((axes[0], None, "d", "number of test images"),
                             (axes[1], "true", ".0%", "share of each true class (rows = 100 %)")):
    ConfusionMatrixDisplay.from_predictions(y_test, pred_test, labels=list(range(N_CLASSES)), display_labels=CLASSES,
                                            normalize=norm, values_format=fmt, ax=ax, colorbar=False,
                                            xticks_rotation=45, cmap="Blues")
    ax.set(title=title, xlabel="predicted class", ylabel="true class (your label)")
plt.tight_layout()
fig.savefig(FIG_DIR / f"{SOURCE}_confusion_matrix.png", dpi=110, bbox_inches="tight")
plt.show()
labels = list(range(N_CLASSES))
display(pd.DataFrame({"test images": [int((y_test == c).sum()) for c in labels],
                      "recall": recall_score(y_test, pred_test, labels=labels, average=None, zero_division=0),
                      "precision": precision_score(y_test, pred_test, labels=labels, average=None, zero_division=0)},
                     index=CLASSES).round(2))

In [ ]:
# 3.4d  The mistakes of the chosen model: is the model wrong — or the label?
wrong = np.where(pred_test != y_test)[0]
print(f"{len(wrong)} of {len(TEST)} test images get a different class than their label")
if len(wrong) and IMG_DIR is not None:
    worst = wrong[np.argsort(-P_test[wrong].max(axis=1))][:15]           # the most confident mistakes first
    show_images([IMG_DIR / LAB.relpath[TEST[i]] for i in worst],
                [f"label: {CLASSES[y_test[i]]}\nmodel: {CLASSES[pred_test[i]]} ({P_test[i].max():.0%})" for i in worst],
                ncols=5, size=2.8, suptitle="the most confident mistakes first")

In [ ]:
# 3.4e  (optional) After the choice: how did all models do on the test images? (for curiosity — the choice is made)
display(results_table(test=True).query("kind == 'this view'").drop(columns="kind").reset_index(drop=True))

✏️ **Discuss:** How many of these mistakes are *really* wrong, and how many are images that sit between two classes — or carry the wrong label? A model trained on imperfect labels cannot be better than its labels. Which classes are mixed up most often (confusion matrix)? Would merging them, or defining them differently, make sense?

⏱ *Short of time? Jump to Section 4 — Sections 3.5 and 3.6 are optional.*

## 3.5 (Optional) One view predicts the other: street ↔ sky
Satellite and street view images were taken at the **same points** (same `image_id`). So we can ask a real prediction question: **does the view from above tell us what the street looks like** (or, with `SOURCE = "satellite"`, does the street tell us what the place looks like from above)? The features now come from the *other* view — the SOM that produced your labels never saw them.

In [ ]:
# 3.5  Cross-view: the labels of this view, the features of the OTHER view of the same point
OTHER = {"streetview": "satellite", "satellite": "streetview"}.get(SOURCE)
CROSS = None
if OTHER is None:
    print("Cross-view needs satellite and street view images of the same points — not available for ArchDaily.")
else:
    check_labels_unchanged()
    # the other view's CSVs (its SOM labels are not needed): best from the same Week 8 folder, so the ids match
    other_dir = next((d for d in (WEEK8_DIR, OWN_WEEK8_DIR, SHARED_WEEK8_DIR) if week8_status(d, OTHER)["CSVs"]), None)
    if other_dir is None:
        print(f"No {OTHER} CSVs found -> skipped.")
    else:
        other = load_week8_table(other_dir, OTHER)
        o_int = [c for c in other.columns if c.startswith("int_") and other[c].std() > 0]
        o_emb = [c for c in other.columns if c.startswith("emb_")]
        CROSS = LAB[["image_id"]].merge(other[["image_id"] + o_int + o_emb], on="image_id", how="left")
        has = CROSS[o_emb[0]].notna().to_numpy()
        cross_idx = tuple(part[has[part]] for part in (TRAIN, VAL, TEST))
        print(f"{has.sum()} of {len(LAB)} labelled {SOURCE} images have a {OTHER} image of the same point "
              f"(from {other_dir})")
        if has.sum() < 50 or min(len(part) for part in cross_idx) < 10:
            print("Too few images of the same points (were the two views made in different Week 8 runs?) -> skipped.")
            CROSS = None
    if CROSS is not None:
        for feats, cols in ((f"{OTHER} int", o_int), (f"{OTHER} emb", o_emb)):
            fit_and_record("logistic regression", LogisticRegression(max_iter=3000, class_weight="balanced"), feats,
                           columns=cols, data=CROSS, kind="other view", idx=cross_idx)
            fit_and_record("random forest", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                                   random_state=SEED, n_jobs=-1),
                           feats, scale=False, columns=cols, data=CROSS, kind="other view", idx=cross_idx)
            fit_mlp(feats, columns=cols, data=CROSS, kind="other view", idx=cross_idx)
        board = results_table()
        same = board[(board.kind == "this view") & board.features.isin(["int", "emb"])]
        cross = board[board.kind == "other view"]
        t = pd.concat([same, cross])
        t = t[t.model.isin(["logistic regression", "random forest", "MLP (PyTorch)"])]
        order = [c for c in ["int", "emb", f"{OTHER} int", f"{OTHER} emb"] if c in set(t.features)]
        piv = t.pivot_table(index="model", columns="features", values="val_bal_acc").reindex(columns=order)
        colours = {"int": "#9ecae1", "emb": "#3182bd", f"{OTHER} int": "#fdae6b", f"{OTHER} emb": "#e6550d"}
        ax = piv.plot.bar(figsize=(10, 4), rot=0, color=[colours[c] for c in piv.columns])
        ax.axhline(1 / N_CLASSES, color="black", ls="--", lw=1, label="chance")
        ax.set(ylim=(0, 1), ylabel="validation balanced accuracy", xlabel="",
               title=f"labels of {SOURCE}: features of the same image (blue) vs of the {OTHER} image (orange)")
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
        plt.tight_layout()
        plt.show()
        # the final exam for the best cross-view model
        best_cross = cross.iloc[0]
        cross_cols = o_int if best_cross.features.endswith("int") else o_emb
        te = cross_idx[2]
        pred = MODELS[(best_cross.model, best_cross.features)].predict(CROSS.loc[te, cross_cols])
        print(f"best cross-view model: {best_cross.model} on '{best_cross.features}' — TEST images: "
              f"accuracy {accuracy_score(Y[te], pred):.2f}, balanced accuracy {balanced_accuracy_score(Y[te], pred):.2f} "
              f"(chance {1 / N_CLASSES:.2f})")
        recall = {c: (pred[Y[te] == i] == i).mean() if (Y[te] == i).any() else np.nan for i, c in enumerate(CLASSES)}
        display(pd.Series(recall).to_frame("share recognised (test)").T.round(2))

✏️ **Discuss:** The scores are lower than with the image's own features — but are they above chance (dashed line)? Which of your classes can be recognised from the other view, which cannot? What does a satellite image show that a street view does not — and the other way round?

## 3.6 (Optional) Preview of next week: learning directly from the pixels
All models so far used numbers that **we** computed last week (VGG16, SegFormer, edges, colour). A **convolutional neural network (CNN)** learns its own features from the pixels: small filters (3 × 3 pixels) slide over the image and react to edges, colours and textures; deeper layers combine them into larger patterns. VGG16 is a big CNN (138 million weights, trained on 1.2 million photos). Here we train a **tiny** one from scratch on our few hundred images, and compare it with an MLP that gets the raw pixels.
How CNNs work in detail — and how to re-use big pre-trained ones — is next week's topic.

In [ ]:
# 3.6a  Small copies of the images (64 × 64 pixels), cached in Week9/cache/ after the first run (~1 minute from Drive)
THUMB = 64


def load_thumbnails(table, size=THUMB):
    cache = CACHE_DIR / f"{SOURCE}_pixels{size}.npz"
    ids = np.asarray(table.image_id.astype(str), dtype=str)          # plain text array (re-loadable without pickle)
    if cache.exists():
        try:
            with np.load(cache) as z:
                if np.array_equal(z["image_id"], ids):
                    return z["pixels"], z["found"]
        except Exception:                                             # unreadable or outdated cache -> rebuild it
            pass
    pixels = np.zeros((len(table), size, size, 3), dtype=np.uint8)
    found = np.zeros(len(table), dtype=bool)
    for i, rel in enumerate(tqdm(table.relpath, desc="reading images")):
        try:
            img = load_rgb(IMG_DIR / rel)
        except Exception:
            continue
        if SOURCE in ("satellite", "streetview"):
            img = img.crop((0, 0, img.width, img.height - GOOGLE_LOGO_PX))          # cut off the Google logo
        pixels[i] = np.asarray(img.resize((size, size), Image.BILINEAR))
        found[i] = True
    np.savez_compressed(cache, image_id=ids, pixels=pixels, found=found)
    return pixels, found


def check_pixels():
    check_labels_unchanged()
    if PIX_STAMP != SPLIT_ID:
        raise RuntimeError("The labels or the split changed: run 3.6a again first.")


PIXELS = None
if IMG_DIR is None:
    print("No images found -> Section 3.6 is skipped.")
else:
    check_labels_unchanged()
    PIX_STAMP = SPLIT_ID
    all_pixels, all_found = load_thumbnails(DATA)
    row_of = pd.Series(np.arange(len(DATA)), index=DATA.image_id.astype(str))
    rows = row_of[LAB.image_id.astype(str)].to_numpy()
    PIXELS, FOUND = all_pixels[rows], all_found[rows]
    PIX_IDX = tuple(part[FOUND[part]] for part in (TRAIN, VAL, TEST))
    print(f"{FOUND.sum()} of {len(LAB)} labelled images loaded -> array {PIXELS.shape} (images × height × width × RGB)")
    plt.figure(figsize=(14, 1.6))
    plt.imshow(np.concatenate(list(PIXELS[:12]), axis=1))
    plt.axis("off")
    plt.title("what the networks below get to see: 64 × 64 pixels")
    plt.show()

In [ ]:
# 3.6b  An MLP on raw pixels: 32 × 32 × 3 = 3,072 numbers per image — it has no idea which pixels are neighbours
if PIXELS is not None:
    check_pixels()
    small = PIXELS.reshape(len(PIXELS), 32, 2, 32, 2, 3).mean(axis=(2, 4)) / 255.0 - 0.5      # 64×64 -> 32×32
    flat = small.reshape(len(PIXELS), -1).astype(np.float32)
    tr, va, te = PIX_IDX
    t0 = time.time()
    torch.manual_seed(SEED)
    net, history, best = train_network(make_mlp(flat.shape[1], N_CLASSES, hidden=(128,), dropout=0.3),
                                       flat[tr], Y[tr], flat[va], Y[va], epochs=150, patience=20, desc="MLP on pixels")
    record("MLP on raw pixels", "pixels 32×32", NetworkClassifier(net), flat[va], flat[te], time.time() - t0,
           kind="pixels", val_idx=va, test_idx=te)
    plot_history(history, best, "MLP on raw pixels")

In [ ]:
# 3.6c  A tiny convolutional network, trained from scratch on the 64 × 64 pictures
class TinyCNN(nn.Module):
    """3 × (3×3 convolution -> ReLU -> pooling), then one score per class."""

    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64×64 -> 32×32, 16 filters
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # -> 16×16, 32 filters
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),                    # -> 16×16, 64 filters
            nn.AdaptiveAvgPool2d(1))                                # -> 64 numbers: the average of each map, as for VGG16
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def mirror(xb):
    """Data augmentation: show a random half of the images mirrored left-right."""
    flip = torch.rand(len(xb), 1, 1, 1, device=xb.device) < 0.5
    return torch.where(flip, xb.flip(3), xb)


if PIXELS is not None:
    check_pixels()
    tr, va, te = PIX_IDX
    x = PIXELS.astype(np.float32).transpose(0, 3, 1, 2) / 255.0                       # (images, 3 colours, 64, 64)
    mean, std = x[tr].mean(axis=(0, 2, 3), keepdims=True), x[tr].std(axis=(0, 2, 3), keepdims=True)
    x = (x - mean) / std
    torch.manual_seed(SEED)
    cnn = TinyCNN(N_CLASSES)
    print(cnn)
    print(f"{sum(p.numel() for p in cnn.parameters()):,} weights (VGG16: 138,000,000)")
    t0 = time.time()
    cnn, history, best = train_network(cnn, x[tr], Y[tr], x[va], Y[va], epochs=40, lr=2e-3, patience=10,
                                       augment=mirror, desc="tiny CNN")
    record("tiny CNN (from scratch)", "pixels 64×64", NetworkClassifier(cnn), x[va], x[te], time.time() - t0,
           kind="pixels", val_idx=va, test_idx=te)
    plot_history(history, best, "tiny CNN")
    board = results_table(test=True)
    best_csv = board[(board.kind == "this view") & (board.features != "–")].head(1)
    display(pd.concat([board[board.kind == "pixels"], best_csv])[["model", "features", "val_bal_acc", "test_bal_acc"]]
            .reset_index(drop=True))

**What did we see?** Compare the **test** scores of the two pixel models with the best model of 3.4 (last row). (The validation score of a network can be quite optimistic: the validation images also chose the epoch where its training stopped — that is why the final exam matters.) Learning from raw pixels with a few hundred images is hard: the networks have to discover from scratch what VGG16 and SegFormer learnt from millions of photos — which is why the models that use last week's features usually win. **Next week:** how CNNs see (filters, feature maps) and how to re-use a big pre-trained CNN for our own classes (*transfer learning*).

---
# 4 · Wrap-up

In [ ]:
# 4.1  The chosen model (best validation score, among the models on this view's CSV columns) labels every image
check_labels_unchanged()
best_row = best_choice()
BEST_KEY = (best_row.model, best_row.features)
BEST_RESULT = next(r for r in RESULTS if (r["model"], r["features"]) == BEST_KEY)     # all its scores
best_model, best_cols = MODELS[BEST_KEY], FEATURE_SETS[best_row.features]
P = proba_all(best_model, DATA[best_cols])
DATA["split"] = "excluded"
DATA.loc[DATA.label.notna().to_numpy(), "split"] = LAB.split.to_numpy()
DATA["pred_label"] = np.array(CLASSES, dtype=object)[P.argmax(axis=1)]
DATA["pred_confidence"] = P.max(axis=1).round(3)
save_labelled_csv()
print(f"chosen model: {best_row.model} on '{best_row.features}' (balanced accuracy: validation "
      f"{BEST_RESULT['val_bal_acc']:.2f}, test {BEST_RESULT['test_bal_acc']:.2f})")
print(f"-> {LABELLED_CSV}: new columns split, pred_label, pred_confidence")
display(DATA[["image_id", "city", "som_cluster", "group", "label", "split", "pred_label", "pred_confidence"]]
        .rename(columns={"group": "subcluster" if LEVEL == "sub-clusters" else "group"}).head(6))
excluded = np.where(DATA.label.isna())[0]
if len(excluded) and IMG_DIR is not None:
    pick = np.random.default_rng(SEED).choice(excluded, size=min(10, len(excluded)), replace=False)
    show_images([IMG_DIR / DATA.relpath[i] for i in pick],
                [f"excluded ({DATA.group[i]})\nmodel: {DATA.pred_label[i]} ({DATA.pred_confidence[i]:.0%})" for i in pick],
                ncols=5, size=2.6, suptitle="what does the model call the images you excluded?")

In [ ]:
# 4.2  Save the best model with a description, and check that the saved file gives the same answers
import joblib
import sklearn

info = {"source": SOURCE, "model": BEST_KEY[0], "features": BEST_KEY[1], "columns": best_cols, "classes": CLASSES,
        "label_plan": PLAN, "merge_level": LEVEL, "split": SPLIT,
        "test_city": TEST_CITY if SPLIT == "by city" else None,
        "scores": {k: float(BEST_RESULT[k]) for k in ["val_acc", "val_bal_acc", "test_acc", "test_bal_acc"]},
        "week8_folder": str(WEEK8_DIR), "created": datetime.now().isoformat(timespec="seconds"),
        "versions": {"sklearn": sklearn.__version__, "torch": torch.__version__, "pandas": pd.__version__}}
if isinstance(best_model, NetworkClassifier):
    path = MODEL_DIR / f"{SOURCE}_best_model.pt"
    torch.save({"state_dict": best_model.model.state_dict(), **best_model.config}, path)
    joblib.dump(best_model.prep, MODEL_DIR / f"{SOURCE}_best_model_preprocessing.joblib")
    saved = torch.load(path, map_location="cpu")
    net = make_mlp(saved["n_inputs"], N_CLASSES, saved["hidden"], saved["dropout"])
    net.load_state_dict(saved["state_dict"])
    again = NetworkClassifier(net.to(DEVICE), joblib.load(MODEL_DIR / f"{SOURCE}_best_model_preprocessing.joblib"),
                              best_cols)
else:
    path = MODEL_DIR / f"{SOURCE}_best_model.joblib"
    joblib.dump(best_model, path)
    again = joblib.load(path)
(MODEL_DIR / f"{SOURCE}_best_model.json").write_text(json.dumps(info, indent=1))
same = (again.predict(DATA[best_cols]) == best_model.predict(DATA[best_cols])).mean()
print(f"saved {path.name} (+ {SOURCE}_best_model.json) | the re-loaded model gives the same answer for {same:.0%} of the images")

In [ ]:
# 4.3  What is in your Week9 folder now?
for f in sorted(WEEK9_DIR.rglob("*")):
    if f.is_file():
        print(f"{str(f.relative_to(WEEK9_DIR)):55s} {f.stat().st_size / 1e6:7.2f} MB")

## What you have now
| In `Week9/` | Content |
|---|---|
| `labels/<source>_labelled.csv` | last week's CSV columns + `subcluster`, **`label`**, `label_id` (your classes), `split`, `pred_label`, `pred_confidence` |
| `labels/<source>_label_plan.json` | your plan: which group became which class |
| `models/<source>_best_model.*` | the best model + a `.json` description (classes, features, scores) |
| `figures/` | model comparison, confusion matrix |
| `cache/` | the 64 × 64 pixel copies of the images used in 3.6 (can be deleted; rebuilt when needed) |

## Exercises (in class if time is left, or as homework — pick two)
1. **Other classes.** Change the names in the panel (2.1) — merge differently, or exclude a group — then re-run **from 2.3 to the end**. Do fewer classes always give higher scores? (Careful: with fewer classes, guessing also gets easier — chance is 1 / number of classes.) Is a plan with high scores also a *useful* plan?
2. **A city the model has never seen.** In 3.1 set `SPLIT = "by city"` and try different `TEST_CITY` values. Which city is the hardest? Then set `SOURCE = "satellite"` (0.2), run everything again and test on `TEST_CITY = "Tallahassee"`. Look at the test **accuracy** and the confusion matrix: what happens, and why? (Hint: look at 1.6 — what do the satellite groups capture for Tallahassee?)
3. **ArchDaily.** Set `SOURCE = "archdaily"`, merge the sub-clusters into **exterior photo / interior photo / drawing** (exclude a group that mixes everything) and train the models. Which indicators does the decision tree use?
4. **Tune the network.** In 3.3a change `HIDDEN` (e.g. `(16,)` or `(128, 64, 32)`), `DROPOUT` (0 … 0.5), `WEIGHT_DECAY`, `PATIENCE` or `MLP_FEATURES`, then re-run 3.3a and 3.3b. Find the best *validation* score — and only then re-run 3.4e to see the test score. Did the test agree?
5. **Bonus — predict a number.** Supervised learning can also predict quantities (*regression*). Predict the street-level greenery (`int_vegetation` of the street view CSV) from the satellite embedding with `RandomForestRegressor`, and measure how close you get with the R² score (1 = perfect, 0 = no better than always guessing the average). To start:
   ```python
   sv, sat = load_week8_table(WEEK8_DIR, "streetview"), load_week8_table(WEEK8_DIR, "satellite")
   both = sv[["image_id", "int_vegetation"]].merge(sat[["image_id"] + [c for c in sat.columns if c.startswith("emb_")]], on="image_id")
   ```